### Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

import json
import pandas as pd
from datetime import datetime
from data_class.raw_data import RawData
from tqdm.auto import tqdm
from clean_text import clean_text
from format_date import format_date

### Helper functions for cleaning

In [2]:
def clean_date(date: str) -> str:
    """Convert various date formats to ISO format"""

    try:
        # Parse date (format: "December 2, 2025")
        date_obj = datetime.strptime(date, '%B %d, %Y')
        return format_date(date_obj.isoformat())
    except ValueError as e:
        print(f"Could not parse date: {date}")
        raise e

def clean_article(article: RawData):
    """Clean a single article entry"""
    cleaned: RawData = article.copy()
    
    # Clean date
    cleaned['publish_date'] = clean_date(cleaned['publish_date'])
    
    # Clean text fields
    for field in ['title', 'content', 'claim', 'verdict']:
        if field in cleaned and cleaned[field]:
            cleaned[field] = clean_text(cleaned[field])
    
    # Ensure authors is a list
    if 'authors' in cleaned and cleaned['authors'] is None:
        cleaned['authors'] = []

    # Add source bias from: https://mediabiasfactcheck.com/full-fact-uk/
    cleaned["source_bias"] = "LEAST-BIASED"

    # Make other props upper case
    cleaned["source"] = cleaned["source"].upper()
    cleaned["type"] = cleaned["type"].upper()
    
    return cleaned

### Load in dataset

In [3]:
with open('../outputs/poynter-factcheck.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

### Clean 

In [4]:
# Clean all articles
cleaned_data = [clean_article(article) for article in tqdm(data, desc="Cleaning articles")]

Cleaning articles:   0%|          | 0/2555 [00:00<?, ?it/s]

### Convert to DF and inspect

In [5]:
df = pd.DataFrame(cleaned_data)

print(f"\nSample of cleaned data:")
print(df[['title', 'content', 'publish_date', 'source_bias']].head())


Sample of cleaned data:
                                               title  \
0  Will new federal student loan caps affect nurs...   
1  No, whooping cough isn’t just a cold. Cases ar...   
2  Trump says he’s never polled better. But poll ...   
3  Turkey prices may be climbing, but your Thanks...   
4  Trump said tariffs slashed the deficit, but th...   

                                             content  \
0  Reports that a Trump administration change mig...   
1  It’s highly infectious and definitely not a we...   
2  Is President Donald Trump more popular than ev...   
3  Turkeys are supposed to go “gobble-gobble,” no...   
4  As President Donald Trump faced questions abou...   

                publish_date   source_bias  
0  2025-12-02T00:00:00+00:00  LEAST-BIASED  
1  2025-12-01T00:00:00+00:00  LEAST-BIASED  
2  2025-11-26T00:00:00+00:00  LEAST-BIASED  
3  2025-11-25T00:00:00+00:00  LEAST-BIASED  
4  2025-11-21T00:00:00+00:00  LEAST-BIASED  


### Save cleaned data to separate JSON files

In [6]:
output_dir = '../outputs_clean/poynter'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/poynter_factcheck.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print("Cleaning complete! Files saved.")

Cleaning complete! Files saved.
